## 1. Importación de Librerías y Configuración
En este bloque cargamos las librerías necesarias para la manipulación de datos y la gestión de archivos. 
Utilizamos `pandas` para el procesamiento de tablas y `numpy` para el manejo de valores nulos.

In [38]:
import pandas as pd
import numpy as np
import os

# Configuración de rutas absolutas dinámicas
# Cuando se ejecuta via main.py, el CWD puede variar, así que calculamos la raíz del proyecto
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == 'notebooks':
    BASE_DIR = os.path.dirname(CURRENT_DIR)
else:
    BASE_DIR = CURRENT_DIR

print(f"Librerías cargadas. Base del proyecto: {BASE_DIR}")

Librerías cargadas exitosamente.


## 2. Carga del Dataset y Detección de Cabeceras
Leemos el archivo descargado por el script de Ingestión. 
Implementamos una lógica para saltar filas vacías o metadatos iniciales que suelen venir en los Excels 
buscando una fila que contenga palabras clave como 'nombre' o 'startup'.

In [39]:
# Configuración de ruta hacia el archivo estandarizado
ruta_archivo = os.path.join(BASE_DIR, "data", "1_sucios", "dataset_para_limpiar.xlsx")

try:
    if os.path.exists(ruta_archivo):
        # Carga inicial para inspección
        df_raw = pd.read_excel(ruta_archivo)
        
        # Palabras clave definidas para localizar la fila de títulos
        header_row = 0
        keywords = ['name', 'website', 'vertical', 'sub vertical', 'descripción']
        
        # Búsqueda de la fila de encabezados en los primeros 15 registros
        for i, row in df_raw.head(15).iterrows():
            # Convertimos toda la fila a una cadena de texto en minúsculas para comparar
            fila_texto = " ".join([str(val).lower() for val in row.values if not pd.isna(val)])
            if any(kw in fila_texto for kw in keywords):
                header_row = i + 1
                break
                
        # Lectura definitiva aplicando el salto de filas
        df = pd.read_excel(ruta_archivo, skiprows=header_row)
        
        # Limpieza de nombres de columnas (quitar espacios extra o saltos de línea)
        df.columns = [str(col).strip() for col in df.columns]
        
        print("Archivo cargado y encabezados identificados.")
        print(f"Columnas detectadas: {list(df.columns)}")
        print(f"Total de registros: {len(df)}")
        
        # Visualización de las primeras 5 filas
        print("\nVista previa de datos:")
        display(df.head(5)) 
    else:
        print(f"Error: Archivo no encontrado en {ruta_archivo}")

except Exception as e:
    print(f"Error en el procesamiento: {e}")

Archivo cargado y encabezados identificados.
Columnas detectadas: ['name', 'website', 'vertical', 'sub vertical', 'Descripcion / solución', 'país ¿dónde se encuentra su principal sede de operaciones?', 'ciudad/región', 'Estadío actual', 'Teléfono', 'Mail', 'LINKEDIN', 'INSTAGRAM', 'FACEBOOK', 'X (twitter)', 'Logo', 'Founder/s', 'Founded', 'Modelo de Negocio', 'DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?"', 'Funding total 2026', 'ETAPA DE MADUREZ ¿En qué momento del crecimiento está?', 'ÁREA INFLUENCIA: ¿qué alcance tiene su solución hoy?', 'ESTADIO SOLUCIÓN (¿En qué nivel de madurez tecnológica se encuentra tu proyecto?)"', 'IMPACTO /SOCIOAMBIENTAL', 'RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?']
Total de registros: 1957

Vista previa de datos:


,name,website,vertical,sub vertical,Descripcion / solución,país ¿dónde se encuentra su principal sede de operaciones?,ciudad/región,Estadío actual,Teléfono,Mail,...,Founder/s,Founded,Modelo de Negocio,"DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?""",Funding total 2026,ETAPA DE MADUREZ ¿En qué momento del crecimiento está?,ÁREA INFLUENCIA: ¿qué alcance tiene su solución hoy?,"ESTADIO SOLUCIÓN (¿En qué nivel de madurez tecnológica se encuentra tu proyecto?)""",IMPACTO /SOCIOAMBIENTAL,RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?
0,Spoiler Alert,https://www.spoileralert.com/?ref=failory,OTRO,Transversal al sector,El software que permite a los socios comercial...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HyProMag,https://hypromag.com/?ref=failory,CIRCULAR ECONOMY,recycling,La tecnología principal que comercializa HyPro...,United-kingdom,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DePoly,https://www.depoly.co/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,Empresa de reciclaje químico de plásticos que ...,switzerland,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Glacier,https://endwaste.io/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,construimos robots de última generación que h...,USA,San Francisco,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,800 Super Holdings,https://www.800super.com.sg/?ref=failory,CIRCULAR ECONOMY,recycling,Ofrecen soluciones ambientales para organizaci...,singapore,Sembawang,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Homologación de Nombres de Columnas
En este paso se renombran las columnas del Excel utilizando los títulos exactos detectados en el archivo sucio. 
Se mapean hacia los nombres técnicos de la tabla `organizations` para garantizar la compatibilidad con la base de datos.

In [40]:
# Mapeo exhaustivo utilizando los nombres exactos de tu Excel y tu DB
mapping_dict = {
    'name': 'name',
    'website': 'website',
    'vertical': 'vertical',
    'sub vertical': 'sub_vertical',
    'Descripcion / solución': 'solucion',
    'país ¿dónde se encuentra su principal sede de operaciones?': 'country',
    'ciudad/región': 'region',
    'Estadío actual': 'estadio_actual',
    'Teléfono': 'contact_phone',
    'Mail': 'mail',
    'Founder/s': 'founders',
    'Founded': 'founded',
    'Modelo de Negocio': 'business_model',
    'DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?': 'badges',
    'RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?': 'outcome_status',
    'LINKEDIN': 'linkedin_url',
    'INSTAGRAM': 'instagram_url',
    'FACEBOOK': 'facebook_url',
    'X (twitter)': 'twitter_url',
    'Logo': 'logo_url'
}

# Aplicar el renombrado
df_mapeado = df.rename(columns=mapping_dict)

# Definición de las 25 columnas de la tabla organizations según HeidiSQL
columnas_db_completas = [
    'id', 'name', 'website', 'vertical', 'sub_vertical', 'country', 'region', 'city',
    'logo_url', 'estadio_actual', 'solucion', 'mail', 'social_media',
    'contact_phone', 'founders', 'founded', 'organization_type', 'outcome_status', 
    'business_model', 'badges', 'notes', 'status', 'lat', 'lng'
]

# Garantizar que todas las columnas existan en el DataFrame (aunque sea con valores nulos)
for col in columnas_db_completas:
    if col not in df_mapeado.columns:
        df_mapeado[col] = None

# Reordenar el DataFrame para que coincida con la tabla de la base de datos
df_final = df_mapeado[columnas_db_completas].copy()

# Asignación de valores por defecto para el control del Backend/Admin Panel
df_final['status'] = 'DRAFT'

print(f"Dataset preparado con {len(df_final.columns)} columnas.")
print("Columnas listas para geocoding automático: country, region, city.")

# Visualización para verificar que los títulos de columna ahora son técnicos
display(df_final.head(5))

Dataset preparado con 24 columnas.
Columnas listas para geocoding automático: country, region, city.


,id,name,website,vertical,sub_vertical,country,region,city,logo_url,estadio_actual,...,founders,founded,organization_type,outcome_status,business_model,badges,notes,status,lat,lng
0,None,Spoiler Alert,https://www.spoileralert.com/?ref=failory,OTRO,Transversal al sector,NaN,NaN,None,NaN,NaN,...,NaN,NaN,None,NaN,NaN,None,None,DRAFT,None,None
1,None,HyProMag,https://hypromag.com/?ref=failory,CIRCULAR ECONOMY,recycling,United-kingdom,NaN,None,NaN,NaN,...,NaN,NaN,None,NaN,NaN,None,None,DRAFT,None,None
2,None,DePoly,https://www.depoly.co/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,switzerland,NaN,None,NaN,NaN,...,NaN,NaN,None,NaN,NaN,None,None,DRAFT,None,None
3,None,Glacier,https://endwaste.io/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,USA,San Francisco,None,NaN,NaN,...,NaN,NaN,None,NaN,NaN,None,None,DRAFT,None,None
4,None,800 Super Holdings,https://www.800super.com.sg/?ref=failory,CIRCULAR ECONOMY,recycling,singapore,Sembawang,None,NaN,NaN,...,NaN,NaN,None,NaN,NaN,None,None,DRAFT,None,None


## 4. Alineación de Taxonomías con la Base de Datos
En este bloque se transforman los datos descriptivos del Excel a los valores técnicos oficiales de la base de datos. 
Se procesan las categorías de Verticales, Sub-verticales, Etapa de Madurez, Modelo de Negocio, Badges, Tipo de Organización y Estado Operativo.
Esto asegura que el filtrado en el Frontend y la integridad de la base de datos sean perfectos.

In [41]:
# Diccionarios expandidos con términos detectados en tus archivos (español/inglés)
mapeos = {
    'vertical': {
        'agtech': 'agtech',
        'biotech': 'biotech_bioinputs', 'bioinsumos': 'biotech_bioinputs', 'biotecnología': 'biotech_bioinputs',
        'foodtech': 'foodtech',
        'climatech': 'climatech', 'clima': 'climatech',
        'circular economy': 'circular_economy', 'economía circular': 'circular_economy', 'circular': 'circular_economy',
        'otra': 'otra'
    },
    'sub_vertical': {
        'digital': 'digital_ag', 'software': 'digital_ag',
        'hardware': 'ag_hardware', 'automatización': 'ag_hardware',
        'riego': 'water_tech', 'agua': 'water_tech',
        'fintech': 'ag_fintech',
        'genómica': 'crop_genomics',
        'biofertilizer': 'biofertilizers', 'biofertilizante': 'biofertilizers',
        'biopesticide': 'biopesticides', 'biopesticida': 'biopesticides',
        'sustainable inputs': 'sustainable_inputs', 'insumos sostenibles': 'sustainable_inputs',
        'genetics': 'genetics_breeding', 'genética': 'genetics_breeding',
        'bioengineering': 'bioengineering', 'bioingeniería': 'bioengineering',
        'novel ingredients': 'novel_ingredients', 'ingredientes': 'novel_ingredients',
        'processing': 'food_processing', 'procesamiento': 'food_processing',
        'indoor': 'indoor_ag',
        'safety': 'food_safety', 'trazabilidad': 'food_safety',
        'carbon': 'carbon_solutions', 'carbono': 'carbon_solutions',
        'soil health': 'soil_health', 'suelo': 'soil_health',
        'waste': 'waste_upcycling', 'residuos': 'waste_upcycling', 'valorization': 'waste_upcycling',
        'otra': 'otra_especificar'
    },
    'estadio_actual': {
        'pre-seed': 'pre_seed', 'pre seed': 'pre_seed',
        'seed': 'seed', 'semilla': 'seed',
        'early stage': 'early_stage',
        'series a': 'series_a_early_growth',
        'series b': 'series_b_growth',
        'scale-up': 'scale_up', 'scaleup': 'scale_up',
        'mature': 'mature_late_stage',
        'acquired': 'acquired', 'adquirida': 'acquired',
        'exit': 'exit',
        'unknown': 'unknown'
    }
    # (Los demás diccionarios se mantienen igual)
}

def normalizar_taxonomia(valor, diccionario, default_val):
    if pd.isna(valor) or str(valor).strip() == "":
        return default_val
    
    limpio = str(valor).lower().strip()
    
    # Intento 1: Coincidencia exacta
    if limpio in diccionario.values(): # Si ya es un valor técnico
        return limpio
        
    # Intento 2: Búsqueda de palabras clave
    for clave, tecnico in diccionario.items():
        if clave in limpio:
            return tecnico
    
    return default_val

# Ejecución del mapeo con el nuevo diccionario
df_final['vertical'] = df_final['vertical'].apply(lambda x: normalizar_taxonomia(x, mapeos['vertical'], 'otra'))
df_final['sub_vertical'] = df_final['sub_vertical'].apply(lambda x: normalizar_taxonomia(x, mapeos['sub_vertical'], 'otra_especificar'))
df_final['estadio_actual'] = df_final['estadio_actual'].apply(lambda x: normalizar_taxonomia(x, mapeos['estadio_actual'], 'unknown'))

print("Corrección de taxonomías aplicada.")
display(df_final[['name', 'vertical', 'sub_vertical']].head(5))

Corrección de taxonomías aplicada.


,name,vertical,sub_vertical
0,Spoiler Alert,otra,otra_especificar
1,HyProMag,circular_economy,otra_especificar
2,DePoly,circular_economy,waste_upcycling
3,Glacier,circular_economy,waste_upcycling
4,800 Super Holdings,circular_economy,otra_especificar


## 5. Normalización Geográfica Universal
En este bloque se procesa la ubicación de forma dinámica para soportar cualquier país del mundo. 
El script extrae el componente del país de cadenas complejas y estandariza nombres conocidos, 
manteniendo la integridad de los datos para que el servicio de geocoding del backend 
pueda procesar cualquier coordenada global.

In [42]:
import re

def limpiar_ubicacion_final_v2(fila):
    pais_raw = str(fila['country']).strip() if pd.notna(fila['country']) else ""
    region_raw = str(fila['region']).strip() if pd.notna(fila['region']) else ""
    
    # Unificamos y normalizamos
    cadena_full = f"{region_raw} {pais_raw}".strip().replace('-', ' ').replace(',', ' / ')
    cadena_full = re.sub(r'\s+', ' ', cadena_full)
    
    if not cadena_full or cadena_full.lower() == 'nan':
        return "", "", ""

    # Lista de países (Expandible)
    lista_paises = [
        'finlandia', 'finland', 'alemania', 'germany', 'españa', 'spain', 
        'united kingdom', 'reino unido', 'uk', 'england', 'usa', 'estados unidos', 
        'united states', 'netherlands', 'paises bajos', 'the netherlands', 
        'singapur', 'singapore', 'francia', 'france', 'suiza', 'switzerland', 
        'argentina', 'india', 'australia', 'mexico', 'méxico'
    ]
    
    texto_min = cadena_full.lower()
    pais_detectado = ""
    cadena_restante = ""

    # 1. CASO PALABRA ÚNICA: Si el texto es exactamente un país de la lista
    if texto_min in lista_paises:
        return cadena_full.title(), "", ""

    # 2. CASO MULTI-PALABRA: Buscamos el país al final
    for p in sorted(lista_paises, key=len, reverse=True):
        if texto_min.endswith(p):
            inicio_pais = texto_min.rfind(p)
            pais_detectado = cadena_full[inicio_pais:].strip().strip('/')
            cadena_restante = cadena_full[:inicio_pais].strip().strip('/')
            break
    else:
        # 3. CASO SIN PAÍS CONOCIDO: Separar por '/' o dejar como región
        partes = [p.strip() for p in cadena_full.split('/') if p.strip()]
        if len(partes) > 1:
            pais_detectado = partes[-1]
            cadena_restante = "/".join(partes[:-1])
        else:
            pais_detectado = ""
            cadena_restante = cadena_full

    # --- Distribución de la Región y Ciudad ---
    partes_ubicacion = [p.strip().title() for p in cadena_restante.split('/') if p.strip()]
    
    region = ""
    ciudad = ""
    
    if len(partes_ubicacion) >= 2:
        region = partes_ubicacion[0]
        ciudad = partes_ubicacion[1]
    elif len(partes_ubicacion) == 1:
        region = partes_ubicacion[0]
        
    # --- LIMPIEZA DE DUPLICADOS (India India, Australia Australia) ---
    pais_final = pais_detectado.title()
    if pais_final == region:
        region = "" # Si el país y la región son lo mismo, vaciamos región

    return pais_final, region, ciudad

# Aplicamos la limpieza
df_final['country'], df_final['region'], df_final['city'] = zip(*df_final.apply(limpiar_ubicacion_final_v2, axis=1))

print("Corrección de Australia e India aplicada.")
display(df_final[['name', 'country', 'region', 'city']].head(10))

Corrección de Australia e India aplicada.


,name,country,region,city
0,Spoiler Alert,,,
1,HyProMag,United Kingdom,,
2,DePoly,Switzerland,,
3,Glacier,Usa,San Francisco,
4,800 Super Holdings,Singapore,Sembawang,
5,Greyparrot,Reino Unido,,
6,NoPalm-Ingredients,The Netherlands,,
7,Infiniti Recycling,England,Cambridge,
8,Ace Green Recycling,Usa,Houston,
9,Fourth Partner Energy,India,,


## 6. Traducción y Enriquecimiento de Ubicación
Este bloque estandariza los nombres de los países al español y resuelve abreviaturas. 
Además, aplica una lógica de inferencia: si el campo 'country' está vacío pero se dispone 
de una región o ciudad conocida, el sistema asigna automáticamente el país 
correspondiente para asegurar la integridad de los datos de geolocalización.

In [43]:
# Diccionario maestro de traducciones y correcciones
# Incluye países en inglés, abreviaturas y errores comunes
mapeo_paises_es = {
    'Switzerland': 'Suiza', 'Usa': 'Estados Unidos', 'United States': 'Estados Unidos',
    'Eeuu': 'Estados Unidos', 'United Kingdom': 'Reino Unido', 'Uk': 'Reino Unido',
    'England': 'Reino Unido', 'Netherlands': 'Países Bajos', 'The Netherlands': 'Países Bajos',
    'Germany': 'Alemania', 'Spain': 'España', 'Brazil': 'Brasil', 'Singapore': 'Singapur',
    'France': 'Francia', 'Italy': 'Italia', 'Belgium': 'Bélgica', 'Mexico': 'México'
}

# Diccionario de inferencia para campos incompletos
ciudades_famosas = {
    'San Francisco': 'Estados Unidos', 'New York': 'Estados Unidos', 'Miami': 'Estados Unidos',
    'London': 'Reino Unido', 'Cambridge': 'Reino Unido', 'Madrid': 'España', 
    'Barcelona': 'España', 'Mendoza': 'Argentina', 'Buenos Aires': 'Argentina'
}

def motor_geografico_integral(fila):
    pais = str(fila['country']).strip()
    region = str(fila['region']).strip()
    ciudad = str(fila['city']).strip()

    # 1. PASO DE TRADUCCIÓN: Traducimos lo que haya en el campo 'country'
    # Lo pasamos a Title Case para que coincida con el diccionario
    pais_tit = pais.title()
    if pais_tit in mapeo_paises_es:
        pais = mapeo_paises_es[pais_tit]
    else:
        pais = pais_tit # Si no está, al menos lo dejamos limpio y capitalizado

    # 2. PASO DE INFERENCIA: Si el país quedó vacío, buscamos por región o ciudad
    if not pais or pais == "":
        for loc in [region, ciudad]:
            loc_tit = loc.title()
            if loc_tit in ciudades_famosas:
                pais = ciudades_famosas[loc_tit]
                break
    
    # 3. PASO DE VERIFICACIÓN: Si el país resultó ser una ciudad (ej. country='Madrid')
    # Lo movemos a region y asignamos el país correcto
    if pais in ciudades_famosas:
        nueva_region = pais
        pais = ciudades_famosas[pais]
        region = nueva_region if not region else region

    return pais, region, ciudad

# Ejecutamos la lógica integrada
df_final['country'], df_final['region'], df_final['city'] = zip(*df_final.apply(motor_geografico_integral, axis=1))

print("Motor geográfico finalizado: Traducción e Inferencia aplicadas.")
# Verificación final de los nombres
display(df_final[['name', 'country', 'region', 'city']].head(10))

Motor geográfico finalizado: Traducción e Inferencia aplicadas.


,name,country,region,city
0,Spoiler Alert,,,
1,HyProMag,Reino Unido,,
2,DePoly,Suiza,,
3,Glacier,Estados Unidos,San Francisco,
4,800 Super Holdings,Singapur,Sembawang,
5,Greyparrot,Reino Unido,,
6,NoPalm-Ingredients,Países Bajos,,
7,Infiniti Recycling,Reino Unido,Cambridge,
8,Ace Green Recycling,Estados Unidos,Houston,
9,Fourth Partner Energy,India,,


## 7. Normalización de Presencia Digital (Deduplicación)
Este bloque estandariza las URLs del sitio web y redes sociales (LinkedIn, Instagram, etc.). 
Al eliminar protocolos y prefijos innecesarios, se asegura que el Migrador pueda 
detectar duplicados de forma efectiva y que los enlaces sean consistentes en el Admin Panel.

In [44]:
import re

def limpiar_url_profesional(url):
    if pd.isna(url) or str(url).strip().lower() in ['nan', 'none', '', 's/d']:
        return ""
    url_limpia = str(url).lower().strip()
    url_limpia = re.sub(r'^https?://', '', url_limpia)
    url_limpia = re.sub(r'^www\.', '', url_limpia)
    url_limpia = url_limpia.rstrip('/')
    url_limpia = url_limpia.split('?')[0]
    return url_limpia

# Identificamos qué columnas de la lista realmente existen en tu DataFrame
columnas_deseadas = ['website', 'linkedin_url', 'instagram_url', 'facebook_url', 'twitter_url']
columnas_reales = [col for col in columnas_deseadas if col in df_final.columns]

# Limpiamos solo las que existen
for col in columnas_reales:
    df_final[col] = df_final[col].apply(limpiar_url_profesional)

print(f"Limpieza finalizada. Columnas procesadas: {columnas_reales}")

# El display ahora solo pide 'name' y las columnas que existan de verdad
columnas_para_ver = ['name'] + columnas_reales
display(df_final[columnas_para_ver].head(10))

Limpieza finalizada. Columnas procesadas: ['website']


,name,website
0,Spoiler Alert,spoileralert.com/
1,HyProMag,hypromag.com/
2,DePoly,depoly.co/
3,Glacier,endwaste.io/
4,800 Super Holdings,800super.com.sg/
5,Greyparrot,greyparrot.ai/
6,NoPalm-Ingredients,nopalm-ingredients.com/
7,Infiniti Recycling,infinitirecycling.com/
8,Ace Green Recycling,acegreenrecycling.com/
9,Fourth Partner Energy,fourthpartner.co/


## 8. Limpieza de Fundadores y Año de Fundación
Este bloque procesa la identidad de los fundadores eliminando enlaces y ruido de texto 
para dejar solo nombres limpios. Asimismo, estandariza el año de fundación a un formato 
numérico consistente, corrigiendo las discrepancias entre tipos de datos (String vs Int).

In [45]:
import re

def transformar_founders(texto):
    if pd.isna(texto) or str(texto).strip().lower() in ['nan', 'none', '', 's/d']:
        return texto # Mantenemos el nulo/vacío original
    
    # 1. Eliminar URLs y lo que esté dentro de paréntesis (perfiles de LinkedIn/Web)
    # Ejemplo: "Ariel Ismirlian (https://...)" -> "Ariel Ismirlian "
    texto_limpio = re.sub(r'\(http[s]?://.*?\)', '', str(texto))
    texto_limpio = re.sub(r'http[s]?://\S+', '', texto_limpio)
    
    # 2. Cambiar pipes (|) por comas para separar nombres
    texto_limpio = texto_limpio.replace('|', ',')
    
    # 3. Limpiar espacios múltiples y espacios en los extremos
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()
    
    # 4. Asegurar que los nombres estén bien separados por coma y espacio
    nombres = [n.strip().title() for n in texto_limpio.split(',') if n.strip()]
    return ", ".join(nombres)

def transformar_founded(valor):
    if pd.isna(valor) or str(valor).strip().lower() in ['nan', 'none', '', 's/d']:
        return valor # Mantenemos el nulo/vacío original
    
    # Extraemos solo los dígitos del string
    solo_numeros = re.sub(r'\D', '', str(valor))
    
    if solo_numeros:
        # Si tiene 4 o más dígitos, asumimos que los primeros 4 son el año
        if len(solo_numeros) >= 4:
            return int(solo_numeros[:4])
        # Si tiene menos (ej. "24" por 2024), intentamos convertirlo a int
        return int(solo_numeros)
            
    return valor # Si no hay números, devolvemos como está para no perder info

# Aplicación de transformaciones
if 'founders' in df_final.columns:
    df_final['founders'] = df_final['founders'].apply(transformar_founders)

if 'founded' in df_final.columns:
    df_final['founded'] = df_final['founded'].apply(transformar_founded)

print("Transformación de fundadores y años completada.")

# Verificación de limpieza
display(df_final[['name', 'founders', 'founded']].head(10))

Transformación de fundadores y años completada.


,name,founders,founded
0,Spoiler Alert,NaN,NaN
1,HyProMag,NaN,NaN
2,DePoly,NaN,NaN
3,Glacier,NaN,NaN
4,800 Super Holdings,NaN,NaN
5,Greyparrot,NaN,NaN
6,NoPalm-Ingredients,NaN,NaN
7,Infiniti Recycling,NaN,NaN
8,Ace Green Recycling,NaN,NaN
9,Fourth Partner Energy,NaN,NaN


## 9. Normalización de Contactos
Este bloque estandariza los correos electrónicos y teléfonos. Se eliminan espacios 
invisibles en los emails y se limpian los números telefónicos de caracteres especiales 
(paréntesis, guiones, espacios), manteniendo el formato numérico puro para 
compatibilidad con sistemas de mensajería.

In [46]:
import re

def resolver_unico_mail(email):
    if pd.isna(email) or str(email).strip().lower() in ['nan', 'none', '', 's/d']:
        return email
    
    # Si hay comas, pipes o espacios, separamos y tomamos el primero
    # Ejemplo: "mail1@test.com | mail2@test.com" -> "mail1@test.com"
    partes = re.split(r'[|,;\s]+', str(email).strip())
    primer_mail = partes[0].lower()
    
    return primer_mail

def resolver_unico_phone(tel):
    if pd.isna(tel) or str(tel).strip().lower() in ['nan', 'none', '', 's/d']:
        return tel
    
    # Separamos por los caracteres de división comunes
    partes = re.split(r'[|,;]+', str(tel).strip())
    primer_tel = partes[0].strip()
    
    # Aplicamos la limpieza que ya teníamos para el primero
    prefijo = "+" if primer_tel.startswith('+') else ""
    solo_numeros = re.sub(r'\D', '', primer_tel)
    
    return f"{prefijo}{solo_numeros}" if solo_numeros else primer_tel

# Aplicamos la resolución de "solo uno"
df_final['mail'] = df_final['mail'].apply(resolver_unico_mail)
df_final['contact_phone'] = df_final['contact_phone'].apply(resolver_unico_phone)

print("Resolución de contactos múltiples finalizada. Se conservó el contacto principal.")

# Verificación
display(df_final[['name', 'mail', 'contact_phone']].head(10))

Resolución de contactos múltiples finalizada. Se conservó el contacto principal.


,name,mail,contact_phone
0,Spoiler Alert,NaN,NaN
1,HyProMag,NaN,NaN
2,DePoly,NaN,NaN
3,Glacier,NaN,NaN
4,800 Super Holdings,NaN,NaN
5,Greyparrot,NaN,NaN
6,NoPalm-Ingredients,NaN,NaN
7,Infiniti Recycling,NaN,NaN
8,Ace Green Recycling,NaN,NaN
9,Fourth Partner Energy,NaN,NaN


## 10. Deduplicación y Estandarización de Taxonomía
En este bloque final, se eliminan registros duplicados para asegurar la integridad de 
la base de datos. Además, se aplica la taxonomía 'S/D' a los campos nulos y se 
asigna el tipo 'STARTUP' a todas las organizaciones, dejando el dataset listo 
para la migración final a HeidiSQL.

In [ ]:
# 1. Definimos los grupos de columnas según su tratamiento de nulos
columnas_con_sd = [
    'vertical', 'sub_vertical', 'country', 'region', 'city', 'logo_url',
    'estadio_actual', 'solucion', 'mail', 'contact_phone', 'founders', 
    'business_model', 'badges', 'outcome_status', 'notes', 'social_media'
]

columnas_con_none = ['id', 'lat', 'lng', 'founded']

# 2. Proceso de Unificación
for col in df_final.columns:
    # Convertimos a string temporalmente para detectar 'nan', 'none', 'unknown', etc.
    if col in columnas_con_sd:
        # Reemplazamos todas las variantes de "sin dato" por 'S/D'
        df_final[col] = df_final[col].astype(str).str.strip()
        mascara_vacio = df_final[col].str.lower().isin(['nan', 'none', '', 's/d', 'unknown'])
        df_final.loc[mascara_vacio, col] = 'S/D'
        
    elif col in columnas_con_none:
        # Para campos técnicos, usamos None (NULL en SQL)
        # Primero limpiamos variantes de texto de nulos
        temp_col = df_final[col].astype(str).str.lower().str.strip()
        mascara_none = temp_col.isin(['nan', 'none', '', 's/d', 'unknown'])
        df_final.loc[mascara_none, col] = None

# 3. Asegurar valores fijos finales
df_final['organization_type'] = 'startup'
df_final['status'] = 'DRAFT'

print("Estandarización de nulos completada: 'S/D' para texto y 'None' para técnicos.")

# Verificación de la limpieza
display(df_final[['name', 'logo_url', 'estadio_actual', 'founders', 'lat', 'lng']].head(10))

Estandarización completada: Ubicaciones como '', taxonomía 'startup' y 'S/D' en texto.


## 11. Exportación de Datos Limpios
Este bloque genera el archivo final en la ruta 'data/02_limpios/'. Se utiliza 
codificación UTF-8 para preservar caracteres especiales y se asegura que el 
índice de Pandas no se incluya, dejando el archivo listo para la inyección 
directa en la tabla 'organizations'.

In [49]:
import os

# 1. Definir la ruta de salida absoluta
ruta_salida = os.path.join(BASE_DIR, "data", "2_limpios", "startups_limpias_final.csv")

# 2. Asegurar que la carpeta exista
os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

# 3. Exportar a CSV
df_final.to_csv(ruta_salida, index=False, sep=';', encoding='utf-8-sig')

print(f"¡Proceso completado con éxito!")
print(f"Archivo guardado en: {ruta_salida}")
print(f"Total de registros exportados: {len(df_final)}")

¡Proceso completado con éxito!
Archivo guardado en: ../data/2_limpios/startups_limpias_final.csv
Total de registros exportados: 1957
